In [2]:
import numpy as np
import torch
import cma

def get_fitness_quadratic(params: np.ndarray, data: torch.Tensor, device: torch.device) -> float:
    """
    Quadratic fitness function with both linear and quadratic terms.
    
    Args:
        params: Parameter vector of shape (D + D*D,) where:
                - First D elements are linear weights
                - Remaining D*D elements are quadratic interaction matrix (flattened)
        data: Input data of shape (N, 2, D)
        device: Target device for computation
    
    Returns:
        Classification accuracy as fitness score
    """
    D = data.shape[2]  # Feature dimension
    
    # Convert to tensor
    params_tensor = torch.tensor(params, dtype=torch.float32, device=device)
    
    # Split parameters
    w_linear = params_tensor[:D]  # Linear weights: shape (D,)
    w_quad_flat = params_tensor[D:]  # Quadratic weights: shape (D*D,)
    w_quad = w_quad_flat.reshape(D, D)  # Reshape to matrix: shape (D, D)
    
    with torch.no_grad():
        # Linear term: data @ w_linear -> (N, 2)
        U_linear = torch.matmul(data, w_linear)
        
        # Quadratic term: data @ W_quad @ data^T (for each sample)
        # First: data @ W_quad -> (N, 2, D)
        data_weighted = torch.matmul(data, w_quad)
        # Then: element-wise multiply and sum -> (N, 2)
        U_quadratic = torch.sum(data * data_weighted, dim=2)
        
        # Combine linear and quadratic terms
        U = U_linear + U_quadratic
    
    # Binary classification accuracy (class 0 vs class 1)
    correct = (U[:, 0] > U[:, 1])
    accuracy = correct.float().mean()
    
    return accuracy.item()

def optimize_quadratic_cmaes(data: torch.Tensor, device: torch.device, 
                           sigma0: float = 0.5, maxiter: int = 1000,
                           popsize: int = None, seed: int = 42):
    """
    Optimize the quadratic fitness function using CMA-ES.
    
    Args:
        data: Input data of shape (N, 2, D)
        device: Device for computation
        sigma0: Initial step size for CMA-ES
        maxiter: Maximum iterations
        popsize: Population size (None for automatic)
        seed: Random seed
        
    Returns:
        Tuple of (best_params, best_fitness, evolution_strategy)
    """
    D = data.shape[2]
    param_dim = D + D * D  # Linear + quadratic parameters
    
    print(f"Data shape: {data.shape}")
    print(f"Feature dimension D: {D}")
    print(f"Total parameters: {param_dim} (Linear: {D}, Quadratic: {D*D})")
    
    # Initialize CMA-ES
    x0 = np.random.randn(param_dim) * 0.1  # Small random initialization
    
    # Create fitness function with data bound
    def fitness_func(params):
        return -get_fitness_quadratic(params, data, device)  # Negative for minimization
    
    es = cma.CMAEvolutionStrategy(x0, sigma0, {
        'maxiter': maxiter,
        'popsize': popsize,
        'seed': seed,
        'verb_disp': 100  # Display every 100 iterations
    })
    
    print("Starting CMA-ES optimization...")
    
    # Optimization loop
    iteration = 0
    while not es.stop():
        solutions = es.ask()
        fitness_values = [fitness_func(x) for x in solutions]
        es.tell(solutions, fitness_values)
        
        if iteration % 100 == 0:
            best_fitness = -min(fitness_values)
            print(f"Iteration {iteration}: Best fitness = {best_fitness:.6f}")
        
        iteration += 1
    
    # Get best solution
    best_params = es.result.xbest
    best_fitness = -es.result.fbest
    
    print(f"\nOptimization completed!")
    print(f"Best fitness: {best_fitness:.6f}")
    print(f"Total iterations: {iteration}")
    
    return best_params, best_fitness, es

def analyze_solution(best_params: np.ndarray, D: int):
    """
    Analyze the optimized parameters.
    
    Args:
        best_params: Optimized parameter vector
        D: Feature dimension
    """
    w_linear = best_params[:D]
    w_quad = best_params[D:].reshape(D, D)
    
    print("\n" + "="*50)
    print("SOLUTION ANALYSIS")
    print("="*50)
    
    print(f"\nLinear weights (shape {w_linear.shape}):")
    print(f"  Mean: {w_linear.mean():.6f}")
    print(f"  Std:  {w_linear.std():.6f}")
    print(f"  Range: [{w_linear.min():.6f}, {w_linear.max():.6f}]")
    
    print(f"\nQuadratic matrix (shape {w_quad.shape}):")
    print(f"  Mean: {w_quad.mean():.6f}")
    print(f"  Std:  {w_quad.std():.6f}")
    print(f"  Range: [{w_quad.min():.6f}, {w_quad.max():.6f}]")
    print(f"  Frobenius norm: {np.linalg.norm(w_quad):.6f}")
    
    # Check if quadratic matrix has structure
    diagonal_dominance = np.abs(np.diag(w_quad)).sum() / np.abs(w_quad).sum()
    print(f"  Diagonal dominance: {diagonal_dominance:.3f}")
    
    symmetry = np.linalg.norm(w_quad - w_quad.T) / np.linalg.norm(w_quad)
    print(f"  Asymmetry ratio: {symmetry:.6f}")



In [1]:
import torch
data = torch.load("data_tensor_2.pt")

In [5]:
# Example usage
if __name__ == "__main__":
    
    device = "cpu"
    print(f"Using device: {device}")
    
    # Run optimization
    best_params, best_fitness, es = optimize_quadratic_cmaes(
        data=data,
        device=device,
        sigma0=0.5,
        maxiter=500,
        seed=42
    )
    
    # Analyze results
    analyze_solution(best_params, D)
    
    # Test the solution
    print(f"\nFinal fitness evaluation: {get_fitness_quadratic(best_params, data, device):.6f}")

Using device: cpu
Data shape: torch.Size([1749370, 2, 23])
Feature dimension D: 23
Total parameters: 552 (Linear: 23, Quadratic: 529)
(11_w,22)-aCMA-ES (mu_w=6.5,w_1=26%) in dimension 552 (seed=42, Thu Sep 18 18:22:54 2025)
Starting CMA-ES optimization...
Iteration 0: Best fitness = 0.629025
Iteration 100: Best fitness = 0.785185
Iteration 200: Best fitness = 0.786946
Iteration 300: Best fitness = 0.787737
Iteration 400: Best fitness = 0.788219

Optimization completed!
Best fitness: 0.788563
Total iterations: 500


NameError: name 'D' is not defined